# Análise de Clusters Climáticos de Recife (PE) com K-Means

Este notebook executa o pré-processamento, a padronização e o agrupamento não-supervisionado (**K-Means**) dos registros meteorológicos horários da estação de **Recife**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Carregar o dataset do Recife
df = pd.read_csv('df_recife.csv')
print(f"Tamanho original: {df.shape}")
df.head()

## 2. Pré-processamento e Limpeza
- Conversão dos valores com vírgula decimal para float.
- Seleção das features climáticas mais relevantes.
- Remoção de registros nulos.

In [ ]:
features = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
    'TEMPERATURA DO PONTO DE ORVALHO (°C)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

for col in features:
    df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

df_clean = df[features].dropna().copy()
print(f"Registros válidos para treino: {df_clean.shape[0]}")

## 3. Escalonamento e Determinação de K (Método do Cotovelo)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean)

inercia = []
k_vals = range(1, 8)
for k in k_vals:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercia.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(k_vals, inercia, marker='o', color='#2b5c8f')
plt.title('Método do Cotovelo (Elbow Method)')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Inércia')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 4. Treinamento do Modelo K-Means e Interpretação
Agrupamos os horários em **3 clusters** climáticos típicos.

In [ ]:
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df_clean['cluster'] = kmeans.fit_predict(X_scaled)

# Perfil médio de cada cluster
perfil = df_clean.groupby('cluster')[features].mean()
display(perfil)

## 5. Visualização dos Clusters e Centróides

In [ ]:
plt.figure(figsize=(10, 6))
cluster_names = {
    0: 'Cluster 0: Frio / Alta Pressão / Chuva',
    1: 'Cluster 1: Tarde Típica (Quente e Mais Seco)',
    2: 'Cluster 2: Noite/Madrugada (Ameno e Úmido)'
}
palette = {0: '#2b5c8f', 1: '#d95f02', 2: '#1b9e77'}

for c in sorted(df_clean['cluster'].unique()):
    sub = df_clean[df_clean['cluster'] == c]
    plt.scatter(
        sub['TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'],
        sub['UMIDADE RELATIVA DO AR, HORARIA (%)'],
        c=palette[c],
        label=cluster_names.get(c, f'Cluster {c}'),
        alpha=0.35,
        s=25
    )

# Centróides no espaço original das variáveis
centroids_orig = scaler.inverse_transform(kmeans.cluster_centers_)
idx_temp = features.index('TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)')
idx_umid = features.index('UMIDADE RELATIVA DO AR, HORARIA (%)')

plt.scatter(
    centroids_orig[:, idx_temp],
    centroids_orig[:, idx_umid],
    c='red',
    marker='X',
    s=180,
    edgecolors='black',
    label='Centróides'
)

plt.title('Dispersão dos Clusters de Recife (Temperatura vs Umidade)', fontsize=13, fontweight='bold')
plt.xlabel('Temperatura do Ar (°C)')
plt.ylabel('Umidade Relativa do Ar (%)')
plt.legend(frameon=True)
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()